# This Notbook version
Data Prep: 50%(No Label) | 50% (Label) -> sampling each card distribution with Weight sampling

# Preparation

In [ ]:
# 2. Install synchronized versions for Python 3.12 (Feb 2026 stable)
!pip install -q torch torchvision torchaudio
!pip install -q datasets==3.6.0 sympy transformers evaluate sentencepiece accelerate scikit-multilearn pythainlp emoji pandarallel

In [ ]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, load_dataset
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, precision_score, recall_score, hamming_loss
import numpy as np
import warnings
import evaluate
import os
from pandarallel import pandarallel
import seaborn as sns
import matplotlib as mpl
warnings.filterwarnings("ignore")

# Initialize with the number of available CPUs
pandarallel.initialize(progress_bar=True)

In [ ]:
# DO NOT include the brackets, just the string inside the quotes
os.environ['KAGGLE_USERNAME'] = "mysterioucz"
os.environ['KAGGLE_KEY'] = "61589d30f7cb3a67f84e1a3b69b7add9"

# Try the list again
!kaggle competitions list

# EDA

In [ ]:
!ls

In [3]:
df = pd.read_csv('train.csv')

In [4]:
df_test = pd.read_csv('test.csv')

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df_test.head()

In [ ]:
classes = df.columns[2:].values
class2id = {class_: id for id, class_ in enumerate(classes)}
id2class = {id: class_ for class_, id in class2id.items()}
print(classes)


In [ ]:
# Exclude 'comment' column from the label classes
label_counts = df[classes].sum(axis=0).sort_values(ascending=False)
labels_perc = (label_counts/df.shape[0])*100
print(labels_perc)
print(f"total perc {labels_perc.sum(axis=0)}")

## Note
This mean that there are at least 76% of data that dont have any label?

In [ ]:
df.shape[0]

In [ ]:
df["label_count"] = df[classes].sum(axis=1)
df.loc[df["label_count"] == 0, ["comment"]]

# Preprocess


## Clean Data

In [12]:
"""
Copied from thai2transformers (https://github.com/vistec-AI/thai2transformers/blob/master/thai2transformers/preprocess.py)
"""
from typing import Collection, Callable
import re
import html
import emoji
from pythainlp.tokenize import word_tokenize

_TK_UNK, _TK_REP, _TK_WREP, _TK_URL, _TK_END = "<unk> <rep> <wrep> <url> </s>".split()

SPACE_SPECIAL_TOKEN = "<_>"

# Pre-compile regex patterns
_RE_HTML_SPACES = re.compile(r"  +")
_RE_URL = re.compile(r"(http|ftp|https)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?")
_RE_EMPTY_BRACKETS = re.compile(r"\(\)|\{\}|\[\]")
_RE_BRACKETS_PUNCTUATION = re.compile(r"\([^a-zA-Z0-9ก-๙]+\)|\{[^a-zA-Z0-9ก-๙]+\}|\[[^a-zA-Z0-9ก-๙]+\]")
_RE_ARTIFACTS_AFTER_OPEN = re.compile(r"(?<=\()[^a-zA-Z0-9ก-๙]+(?=[a-zA-Z0-9ก-๙])|(?<=\{)[^a-zA-Z0-9ก-๙]+(?=[a-zA-Z0-9ก-๙])|(?<=\[)[^a-zA-Z0-9ก-๙]+(?=[a-zA-Z0-9ก-๙])")
_RE_ARTIFACTS_BEFORE_CLOSE = re.compile(r"(?<=[a-zA-Z0-9ก-๙])[^a-zA-Z0-9ก-๙]+(?=\))|(?<=[a-zA-Z0-9ก-๙])[^a-zA-Z0-9ก-๙]+(?=\})|(?<=[a-zA-Z0-9ก-๙])[^a-zA-Z0-9ก-๙]+(?=\])")
_RE_NEWLINES = re.compile(r"[\n]")
_RE_USELESS_SPACES = re.compile(r" {2,}")
_RE_SPACES = re.compile(r" ")
_RE_REP = re.compile(r"(\S)(\1{3,})")

# str->str rules
def fix_html(text: str) -> str:
    """
        List of replacements from html strings in `test`. (code from `fastai`)
        :param str text: text to replace html string
        :return: text where html strings are replaced
        :rtype: str
        :Example:
            >>> fix_html("Anbsp;amp;nbsp;B @.@ ")
            A & B.
    """
    text = (
        text.replace("#39;", "'")
        .replace("amp;", "&")
        .replace("#146;", "'")
        .replace("nbsp;", " ")
        .replace("#36;", "$")
        .replace("\\n", "\n")
        .replace("quot;", "'")
        .replace("<br />", "\n")
        .replace('\\"', '"')
        .replace(" @.@ ", ".")
        .replace(" @-@ ", "-")
        .replace(" @,@ ", ",")
        .replace("\\", " \\ ")
    )
    return _RE_HTML_SPACES.sub(" ", html.unescape(text))

def replace_url(text: str) -> str:
    """
        Replace url in `text` with TK_URL (https://stackoverflow.com/a/6041965)
        :param str text: text to replace url
        :return: text where urls  are replaced
        :rtype: str
        :Example:
            >>> replace_url("go to https://github.com")
            go to <url>
    """
    return _RE_URL.sub(_TK_URL, text)

def rm_brackets(text: str) -> str:
    """
        Remove all empty brackets and artifacts within brackets from `text`.
        :param str text: text to remove useless brackets
        :return: text where all useless brackets are removed
        :rtype: str
        :Example:
            >>> rm_brackets("hey() whats[;] up{*&} man(hey)")
            hey whats up man(hey)
    """
    # remove empty brackets
    new_line = _RE_EMPTY_BRACKETS.sub("", text)
    # brakets with only punctuations
    new_line = _RE_BRACKETS_PUNCTUATION.sub("", new_line)
    # artifiacts after (
    new_line = _RE_ARTIFACTS_AFTER_OPEN.sub("", new_line)
    # artifacts before )
    new_line = _RE_ARTIFACTS_BEFORE_CLOSE.sub("", new_line)
    return new_line

def replace_newlines(text: str) -> str:
    """
        Replace newlines in `text` with spaces.
        :param str text: text to replace all newlines with spaces
        :return: text where all newlines are replaced with spaces
        :rtype: str
        :Example:
            >>> rm_useless_spaces("hey whats\n\nup")
            hey whats  up
    """
    return _RE_NEWLINES.sub(" ", text.strip())

def rm_useless_spaces(text: str) -> str:
    """
        Remove multiple spaces in `text`. (code from `fastai`)
        :param str text: text to replace useless spaces
        :return: text where all spaces are reduced to one
        :rtype: str
        :Example:
            >>> rm_useless_spaces("oh         no")
            oh no
    """
    return _RE_USELESS_SPACES.sub(" ", text)

def replace_spaces(text: str, space_token: str = SPACE_SPECIAL_TOKEN) -> str:
    """
        Replace spaces with _
        :param str text: text to replace spaces
        :return: text where all spaces replaced with _
        :rtype: str
        :Example:
            >>> replace_spaces("oh no")
            oh_no
    """
    return _RE_SPACES.sub(space_token, text)

def replace_rep_after(text: str) -> str:
    """
    Replace repetitions at the character level in `text`
    :param str text: input text to replace character repetition
    :return: text with repetitive tokens removed.
    :rtype: str
    :Example:
        >>> text = "กาาาาาาา"
        >>> replace_rep_after(text)
        'กา'
    """

    def _replace_rep(m):
        c, cc = m.groups()
        return f"{c}"

    return _RE_REP.sub(_replace_rep, text)

# List[str] -> List[str] rules
def ungroup_emoji(toks: Collection[str]) -> Collection[str]:
    """
    Ungroup Zero Width Joiner (ZVJ) Emojis
    See https://emojipedia.org/emoji-zwj-sequence/
    :param Collection[str] toks: list of tokens
    :return: list of tokens where emojis are ungrouped
    :rtype: Collection[str]
    :Example:
        >>> toks = []
        >>> ungroup_emoji(toks)
        []
    """
    res = []
    for tok in toks:
        if emoji.emoji_count(tok) == len(tok):
            res.extend(list(tok))
        else:
            res.append(tok)
    return res

def replace_wrep_post(toks: Collection[str]) -> Collection[str]:
    """
    Replace reptitive words post tokenization;
    fastai `replace_wrep` does not work well with Thai.
    :param Collection[str] toks: list of tokens
    :return: list of tokens where repetitive words are removed.
    :rtype: Collection[str]
    :Example:
        >>> toks = ["กา", "น้ำ", "น้ำ", "น้ำ", "น้ำ"]
        >>> replace_wrep_post(toks)
        ['กา', 'น้ำ']
    """
    previous_word = None
    rep_count = 0
    res = []
    for current_word in toks + [_TK_END]:
        if current_word == previous_word:
            rep_count += 1
        elif (current_word != previous_word) & (rep_count > 0):
            res += [previous_word]
            rep_count = 0
        else:
            res.append(previous_word)
        previous_word = current_word
    return res[1:]

def remove_space(toks: Collection[str]) -> Collection[str]:
    """
    Do not include space for bag-of-word models.
    :param Collection[str] toks: list of tokens
    :return: Collection of tokens where space tokens (" ") are filtered out
    :rtype: Collection[str]
    :Example:
        >>> toks = ['ฉัน','เดิน',' ','กลับ','บ้าน']
        >>> remove_space(toks)
        ['ฉัน','เดิน','กลับ','บ้าน']
    """
    res = []
    for t in toks:
        t = t.strip()
        if t:
            res.append(t)
    return res

# combine them together
def process_transformers(
    text: str,
    pre_rules: Collection[Callable] = [
        fix_html,
        rm_brackets,
        replace_newlines,
        rm_useless_spaces,
        replace_spaces,
        replace_rep_after,
    ],
    tok_func: Callable = word_tokenize,
    post_rules: Collection[Callable] = [ungroup_emoji, replace_wrep_post],
) -> str:
    text = text.lower()
    for rule in pre_rules:
        text = rule(text)
    return text


In [13]:
def text_clean(text):
    # Ensure input is string
    if not isinstance(text, str):
        return ""

    text = process_transformers(text)

    return text

In [14]:
def clean_data(df):
  df['comment'] = df['comment'].fillna("")
  df['comment'] = df['comment'].parallel_apply(text_clean)
  return df


In [ ]:
df = df.drop(columns=['id'],axis=1) # remove id
df = clean_data(df)
df_test = clean_data(df_test)


In [ ]:
df["label_count"] = df[classes].sum(axis=1)
df.loc[df["label_count"] == 0, ["comment"]]

## Train Test Split

In [17]:
X = df[['comment']].values
y = df[classes].values

In [ ]:
y

In [19]:
from skmultilearn.model_selection import iterative_train_test_split

In [20]:
X_train_arr,y_train_arr ,X_eval_arr, y_eval_arr = iterative_train_test_split(
    X, y, test_size=0.2
)

In [21]:
# 3. Reconstruct DataFrames
df_train = pd.DataFrame(X_train_arr, columns=['comment'])
df_train[classes] = y_train_arr

df_eval = pd.DataFrame(X_eval_arr, columns=['comment'])
df_eval[classes] = y_eval_arr


## Analyze Class Distribution

Calculate and visualize the distribution of labels in the training set to identify minority and majority classes.


**Reasoning**:
Calculate the class distribution by summing the label columns in the training dataframe and visualize it using a bar chart.



In [22]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl

In [23]:
# !wget -q "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf"
font_path = "Sarabun-Regular.ttf"

In [24]:
mpl.font_manager.fontManager.addfont(font_path)
mpl.rc('font', family='Sarabun')

In [25]:
def class_distribution(df_train):
    # Calculate the sum of each class in the training set
    class_counts = df_train[classes].sum().sort_values(ascending=False)
    class_counts["no_label"] = sum(df_train[classes].sum(axis=1) == 0)
    # Print the numerical counts
    print("Class Counts in Training Set:")
    print(class_counts)

    # Create a bar chart
    plt.figure(figsize=(12, 6))
    sns.barplot(x=class_counts.values, y=class_counts.index, palette='viridis')
    plt.title('Class Distribution in Training Set')
    plt.xlabel('Count')
    plt.ylabel('Class')
    plt.show()

In [ ]:
class_distribution(df_train)

**Reasoning**:
The previous step showed a significant class imbalance. Now I need to implement the resampling strategy as requested. This involves creating a function that separates minority and majority classes, oversamples the minority ones, undersamples the majority ones (keeping high multi-label density samples), and then combines them back.



In [ ]:
def balance_dataset_optimized(df, labels, target_total_samples=50000, alpha=0.5, background_ratio=0.15):
    """
    Balances multi-label dataset while maintaining a fixed ratio of 'background' (no-label) examples.
    """
    print(f"Original shape: {df.shape}")
    
    # 1. Identify Labeled vs Unlabeled Data
    # Calculate density to split them apart
    # Make a copy to avoid SettingWithCopy warnings
    df = df.copy()
    df['label_count'] = df[labels].sum(axis=1)
    
    df_labeled = df[df['label_count'] > 0]
    df_background = df[df['label_count'] == 0]
    
    print(f"Labeled samples: {len(df_labeled)}")
    print(f"Background (No Label) samples: {len(df_background)}")

    # 2. Calculate sizes for the final mix
    n_background = int(target_total_samples * background_ratio)
    n_labeled = target_total_samples - n_background
    
    print(f"Target Mix: {n_labeled} Labeled | {n_background} Background")

    # --- PROCESS LABELED DATA (Weighted Sampling) ---
    # 3. Calculate label weights (Inverse Frequency) on the LABELED portion only
    label_counts = df_labeled[labels].sum()
    total_labels = label_counts.sum()
    
    # Rare labels get higher weight
    label_weights = total_labels / (label_counts + 1)
    
    # 4. Assign "Importance Score" to labeled rows
    # We use the MAX weight of the labels present in the row
    # Vectorized: weighted_matrix will have non-zero values where labels exist
    weighted_matrix = df_labeled[labels].multiply(label_weights, axis=1)
    row_scores = weighted_matrix.max(axis=1)
    
    # Apply smoothing alpha
    row_scores = row_scores ** alpha 

    # Sample the labeled portion
    df_balanced_labeled = df_labeled.sample(
        n=n_labeled,
        weights=row_scores,
        replace=True,
        random_state=42
    )

    # --- PROCESS BACKGROUND DATA (Random Sampling) ---
    # 5. Sample the background portion
    if len(df_background) > 0:
        # If we have enough unique background samples, we try not to replace
        replace_bg = len(df_background) < n_background
        df_balanced_background = df_background.sample(
            n=n_background,
            replace=replace_bg,
            random_state=42
        )
    else:
        df_balanced_background = pd.DataFrame(columns=df.columns)

    # 6. Combine and Shuffle
    # Drop the temporary column before concatenating or after
    df_balanced_labeled = df_balanced_labeled.drop(columns=['label_count'])
    df_balanced_background = df_balanced_background.drop(columns=['label_count'])
    
    df_final = pd.concat([df_balanced_labeled, df_balanced_background])
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"Final dataset size: {len(df_final)}")
    # print("Final Label Distribution:\n", df_final[labels].sum())
    
    return df_final

# Try using alpha=0.7 for strong balancing
# Reserve 15% of the dataset for "background" examples so the model learns to predict "nothing"
df_train_balanced = balance_dataset_optimized(
    df_train, 
    classes, 
    target_total_samples=100000, 
    alpha=0.7, 
    background_ratio=0.5
)

In [ ]:
class_distribution(df_train_balanced)

In [29]:

# 4. Convert to Hugging Face Datasets
hf_train = Dataset.from_pandas(df_train_balanced)
hf_eval = Dataset.from_pandas(df_eval)
hf_test = Dataset.from_pandas(df_test)

# 5. Map to DatasetDict
raw_datasets = DatasetDict({
    'train': hf_train,
    'eval': hf_eval,
    'test': hf_test  # Ensure df_test was defined earlier
})

In [30]:
from transformers import AutoTokenizer

model_path = 'clicknext/phayathaibert'

tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
def optimized_preprocess(batches):
    # 1. Tokenize the entire batch of texts at once
    tokenized_inputs = tokenizer(
        batches['comment'],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # 2. Extract labels only if they exist (Train/Eval)
    # We check if the first class column is present in the batch
    if classes[0] in batches:
        batch_labels = []
        for i in range(len(batches['comment'])):
            labels = [float(batches[col][i]) for col in classes]
            batch_labels.append(labels)
        tokenized_inputs['labels'] = batch_labels

    return tokenized_inputs

# Process Train, Eval, and Test
tokenized_train = raw_datasets['train'].map(
    optimized_preprocess,
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)

tokenized_eval = raw_datasets['eval'].map(
    optimized_preprocess,
    batched=True,
    remove_columns=raw_datasets['eval'].column_names
)

# Test set usually doesn't have labels, so the function handles that
tokenized_test = raw_datasets['test'].map(
    optimized_preprocess,
    batched=True,
    remove_columns=raw_datasets['test'].column_names
)


In [32]:
from datasets import DatasetDict

tokenized_datasets = DatasetDict({
    'train': tokenized_train,
    'eval': tokenized_eval,
    'test': tokenized_test
})

In [33]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Calculate Class Weights based on the original training distribution
# This helps the model pay more attention to underrepresented classes
import torch
import numpy as np

# We use the original df_train to calculate true distribution weights
# (Even if we use the balanced dataset for training, checking weights against real distribution is useful,
# but typically we use the weights of the dataset we are actually training on if we want to force balance further,
# or use original weights if we want to counteract the natural imbalance).
# Here we calculate based on the training data we are fed (df_train_balanced or df_train).

# Let's use the df_train (original) to get the "real" difficulty of each class
labels_tensor = torch.tensor(df_train[classes].values, dtype=torch.float)
pos_counts = torch.sum(labels_tensor, dim=0)
neg_counts = len(df_train) - pos_counts

# Formula: pos_weight = neg_counts / pos_counts
# This makes the loss for a positive example of a rare class much larger.
pos_weights = neg_counts / (pos_counts + 1e-5) # 1e-5 to avoid division by zero

print("Class Weights (Pos Weight):", pos_weights)

# Metric

In [35]:
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

best_thresholds = []

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))

    thresholds = getattr(compute_metrics, "thresholds", [0.5] * labels.shape[1])

    y_pred = np.zeros(probs.shape)
    for i in range(labels.shape[1]):
        y_pred[:, i] = (probs[:, i] > thresholds[i]).astype(int)

    # Hamming Loss: misclassified example-label pairs (lower is better)
    h_loss = hamming_loss(labels, y_pred)

    # Subset Accuracy: strict match for all labels (higher is better)
    subset_acc = accuracy_score(labels, y_pred)

    return {
        # Core metrics for imbalanced multi-label data
        "hamming_loss": h_loss,
        "micro_f1": f1_score(labels, y_pred, average='micro', zero_division=0),

        # Macro metrics for balanced class awareness
        "macro_f1": f1_score(labels, y_pred, average='macro', zero_division=0),
        "precision_macro": precision_score(labels, y_pred, average='macro', zero_division=0),
        "recall_macro": recall_score(labels, y_pred, average='macro', zero_division=0),

        # Performance/Categorization metrics
        "subset_accuracy": subset_acc,
        "example_based_accuracy": np.mean(np.all(labels == y_pred, axis=1)) # Strict example accuracy
    }

compute_metrics.thresholds = [0.5] * len(classes)


# Model

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=len(classes),
    id2label=id2class,
    label2id=class2id,
    problem_type="multi_label_classification"
)

##

# Training

In [37]:
os.environ["WANDB_API_KEY"]="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"

In [38]:
from torch import nn
from transformers import Trainer

class MultilabelTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Ensure weights are on the same device as the model
        if class_weights is not None:
            self.class_weights = class_weights.to(self.model.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Binary Cross Entropy with Logits Loss + Class Weights
        if self.class_weights is not None:
             # Ensure weights are on the same device as the model/logits
            if self.class_weights.device != logits.device:
                self.class_weights = self.class_weights.to(logits.device)
            loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.class_weights)
        else:
            loss_fct = nn.BCEWithLogitsLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.float().view(-1, self.model.config.num_labels))

        return (loss, outputs) if return_outputs else loss

In [ ]:
import wandb
# This forcefully stops the wandb background process
wandb.init(project="multi-label-classify", name="weighted-bert-v1")

In [ ]:
# Monitor via wandb
report_to = "wandb"

training_args = TrainingArguments(
    output_dir="my_multi_label_classify_model_weighted",
    learning_rate=3e-5,              # Slightly higher initial LR for BERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # Increase epochs (early stopping will catch it)
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=2000,                  # Evaluate more frequently
    save_steps=2000,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to=report_to,
    warmup_ratio=0.1,
    bf16=True,                       # Keep fp16/bf16
    metric_for_best_model="macro_f1", # Essential for imbalanced data
    lr_scheduler_type="cosine",      # Cosine schedule
    seed=42,
    data_seed=42,
    full_determinism=True,
)

In [ ]:
%%time
# Initialize the Custom Trainer
trainer = MultilabelTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['eval'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=pos_weights  # Pass the calculated weights here
)

trainer.train()

In [ ]:
# Convert logs to a Pandas DataFrame
df_logs = pd.DataFrame(trainer.state.log_history)

# 2. Filter for evaluation logs
# (Evaluation rows have 'eval_loss', whereas training rows have 'loss')
eval_stats = df_logs.dropna(subset=['eval_loss'])

# 3. Print the specific metrics
# Note: your 'f1_macro' becomes 'eval_f1_macro' in the logs
columns_to_show = ['epoch', 'step', 'eval_loss', 'eval_macro_f1', 'eval_accuracy']

# Use .get() or check if columns exist to avoid errors if a metric hasn't run yet
available_columns = [col for col in columns_to_show if col in eval_stats.columns]
print(eval_stats[available_columns])

# Find threshold for each class

In [ ]:
# 1. Get predictions (probabilities) from your trainer
# We ensure we are predicting on the EVALUATION set for threshold tuning
predictions = trainer.predict(tokenized_datasets['eval'])
probs = torch.sigmoid(torch.Tensor(predictions.predictions)).numpy()
y_true = predictions.label_ids

print(f"Probs shape: {probs.shape}")
print(f"True labels shape: {y_true.shape}")

In [ ]:
# 2. Find optimal threshold per class using Grid Search
best_thresholds = []

# Define the grid of thresholds to search (0.01 to 0.99)
threshold_grid = np.linspace(0.01, 0.99, 100)

print("Performing Grid Search for optimal thresholds per class...")
for i in range(len(classes)):
    # Calculate F1 score for all thresholds in the grid for the current class
    f1_scores = [f1_score(y_true[:, i], (probs[:, i] >= t).astype(int), zero_division=0) for t in threshold_grid]

    # Identify the threshold that maximizes the F1 score
    best_idx = np.argmax(f1_scores)
    best_t = threshold_grid[best_idx]
    best_f1 = f1_scores[best_idx]

    best_thresholds.append(best_t)
    print(f"Class: {classes[i]:<40} | Best Threshold: {best_t:.4f} | Max F1: {best_f1:.4f}")

# Update the metrics function with the new best thresholds so future evaluations use them
compute_metrics.thresholds = best_thresholds

In [ ]:
# Check how many actual samples you have per class in the eval set
support = y_true.sum(axis=0)
for i, class_name in enumerate(classes):
    print(f"{class_name}: {support[i]} samples")

In [ ]:
y_pred_eval = probs
for i, threshold in enumerate(best_thresholds):
    y_pred_eval[:, i] = (probs[:, i] > threshold).astype(int) #TODO: Replace 0.5 with custom threshold for each class
print(classification_report(y_true=y_true, y_pred=y_pred_eval, target_names=classes))

# Save model & Config

In [47]:
# filename = input()
filename = "no_label0.5_100k"

In [ ]:
trainer.save_model(f"my_multi_label_classify_model_{filename}")

# Evaluation

In [ ]:
probs.shape

In [ ]:
# 1. Get predictions (logits) from the A100-powered trainer
preds_output = trainer.predict(tokenized_datasets['test'])

# 2. Convert logits to probabilities using Sigmoid
probs = torch.sigmoid(torch.tensor(preds_output.predictions)).numpy()

# 3. Create an empty matrix for binary predictions
y_pred = np.zeros(probs.shape, dtype=int)

# 4. Apply each class-specific threshold
# best_thresholds should be the list you generated earlier
for i, threshold in enumerate(best_thresholds):
    y_pred[:, i] = (probs[:, i] > threshold).astype(int) #TODO: Replace 0.5 with custom threshold for each class

In [ ]:
y_pred.shape

In [ ]:
submit_sample = pd.read_csv('sample_submission.csv')
submit_sample.head()

In [ ]:
prediction_df = pd.DataFrame(y_pred, columns=classes)

# Check if lengths match
if len(prediction_df) == len(submit_sample):
    # Assign predictions to the submission template
    submit_sample[classes] = prediction_df

    # Save to CSV
    submit_sample.to_csv(f"submission_{filename}.csv", index=False)
    print('Submission saved to f"submission_{filename}.csv"')
    display(submit_sample.head())